In [1]:
# ==============================================================================
# 0. DEPENDÊNCIAS E CONFIGURAÇÃO GLOBAL
# ==============================================================================
!pip install -q gspread gspread-dataframe

import pandas as pd
from google.cloud import bigquery
from google.colab import auth
from google.auth import default
import gspread
from gspread_dataframe import set_with_dataframe
import re
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# Autenticação
auth.authenticate_user()
creds, _ = default()

# Configurações
PROJECT_ID = "hip-bonito-453017-m2"
DATASET    = "bronze"
TABELA     = "cigam__empresas"

# Clientes
client_bq = bigquery.Client(project=PROJECT_ID, credentials=creds)
client_gs = gspread.authorize(creds)

def bq_query_to_df(query: str, params=None) -> pd.DataFrame:
    job_config = bigquery.QueryJobConfig(query_parameters=params) if params else None
    rows = client_bq.query(query, job_config=job_config).result()
    return pd.DataFrame([dict(r.items()) for r in rows])

# Paleta da empresa
COR_PRIMARIA   = "#382fd8"   # Roxo/azul
COR_ESCURA     = "#1a1a2e"   # Quase preto
COR_DESTAQUE   = "#19e098"   # Verde água
COR_BRANCO     = "#ffffff"
COR_CINZA      = "#6b7280"

In [2]:
# ==============================================================================
# DIAGNÓSTICO 3 — A BASE ESTÁ DESATUALIZADA?
# ==============================================================================
q_freshness = f"""
SELECT
    MIN(inserted_at)      AS carga_mais_antiga,
    MAX(inserted_at)      AS carga_mais_recente,
    MIN(dataModificacao)  AS modificacao_mais_antiga,
    MAX(dataModificacao)  AS modificacao_mais_recente,
    MIN(dataReferencia)   AS referencia_mais_antiga,
    MAX(dataReferencia)   AS referencia_mais_recente,
    COUNT(*) AS total_linhas
FROM `{PROJECT_ID}.{DATASET}.{TABELA}`
"""
print("🕐 Janela de atualização da tabela:")
display(bq_query_to_df(q_freshness))

# Quantas empresas SEM e-mail em nenhum contato têm modificação recente na origem?
# (se muitas tiverem dataModificacao recente, o cliente cadastrou/editou faz pouco
# tempo e é esperado ainda não ter e-mail; se forem antigas, é lacuna de cadastro mesmo)
q_sem_email_recencia = f"""
SELECT
    dataModificacao,
    COUNT(*) AS qtd_empresas_sem_email
FROM `{PROJECT_ID}.{DATASET}.{TABELA}` e
WHERE e.contatos_json IS NOT NULL AND e.contatos_json != ''
  AND (
      SELECT COUNT(*) FROM UNNEST(JSON_EXTRACT_ARRAY(e.contatos_json)) c
      WHERE JSON_EXTRACT_SCALAR(c, '$.email') IS NOT NULL
        AND JSON_EXTRACT_SCALAR(c, '$.email') != ''
  ) = 0
GROUP BY dataModificacao
ORDER BY dataModificacao DESC
LIMIT 20
"""
print("\n📅 Das empresas sem e-mail, quando foram modificadas pela última vez (top 20 datas mais recentes):")
display(bq_query_to_df(q_sem_email_recencia))

🕐 Janela de atualização da tabela:


,carga_mais_antiga,carga_mais_recente,modificacao_mais_antiga,modificacao_mais_recente,referencia_mais_antiga,referencia_mais_recente,total_linhas
0,2026-01-22 15:19:01.706048+00:00,2026-10-19 08:40:18.586407+00:00,2020-09-25,2026-10-16,2026-01-22,2026-10-19,6694



📅 Das empresas sem e-mail, quando foram modificadas pela última vez (top 20 datas mais recentes):


,dataModificacao,qtd_empresas_sem_email
0,2026-10-16,1
1,2026-10-05,1
2,2026-09-29,1
3,2026-09-23,3
4,2026-09-21,2
5,2026-09-18,1
6,2026-09-09,1
7,2026-09-03,3
8,2026-08-13,1
9,2026-08-06,2


In [3]:
# ==============================================================================
# DIAGNÓSTICO 1 — CAMPO DE ATIVO/INATIVO E CAMPOS DE E-MAIL "SOLTOS"
# ==============================================================================
candidatos_situacao = ['situacao', 'status', 'ativo', 'inativo', 'bloqueado',
                        'flgativo', 'flginativo', 'situacaocadastral',
                        'dtinativacao', 'datainativacao', 'clienteativo',
                        'clienteinativo', 'ativoinativo']
candidatos_email_direto = ['email', 'emailcontato', 'emailprincipal', 'emailnfe',
                            'emailcobranca', 'emailfinanceiro']
candidatos_atualizacao = ['_airbyte_extracted_at', 'dtatualizacao', 'dataatualizacao',
                           'updated_at', 'ultima_atualizacao', 'load_ts', 'etl_ts',
                           '_fivetran_synced']

query_colunas = f"""
SELECT column_name, data_type
FROM `{PROJECT_ID}.{DATASET}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = '{TABELA}'
ORDER BY ordinal_position
"""
df_colunas = bq_query_to_df(query_colunas)
print(f"📋 {len(df_colunas)} colunas na tabela {TABELA}:")
display(df_colunas)

colunas_lower = {c.lower(): c for c in df_colunas['column_name']}

def _achados(lista, titulo):
    achados = [colunas_lower[c] for c in lista if c in colunas_lower]
    print(f"\n🔎 {titulo}: {achados if achados else 'nenhuma coluna óbvia — confira a lista completa acima manualmente'}")
    return achados

achados_situacao   = _achados(candidatos_situacao, "Possíveis colunas de ativo/inativo")
achados_email      = _achados(candidatos_email_direto, "Possíveis colunas de e-mail FORA do contatos_json")
achados_atualizacao = _achados(candidatos_atualizacao, "Possíveis colunas de data de carga/atualização (ETL)")

for col in achados_situacao:
    q = f"""
    SELECT {col} AS valor, COUNT(*) AS qtd
    FROM `{PROJECT_ID}.{DATASET}.{TABELA}`
    GROUP BY {col}
    ORDER BY qtd DESC
    """
    print(f"\n--- Distribuição de valores em '{col}' ---")
    display(bq_query_to_df(q))

for col in achados_atualizacao:
    q = f"""
    SELECT MIN({col}) AS mais_antigo, MAX({col}) AS mais_recente, COUNT(DISTINCT {col}) AS valores_distintos
    FROM `{PROJECT_ID}.{DATASET}.{TABELA}`
    """
    print(f"\n--- Janela de atualização em '{col}' (ajuda a saber se a base está desatualizada) ---")
    display(bq_query_to_df(q))

📋 85 colunas na tabela cigam__empresas:


,column_name,data_type
0,codigo,STRING
1,nomeCompleto,STRING
2,contato,STRING
3,fantasia,STRING
4,pessoa,STRING
...,...,...
80,percentualComissaoBaixa,STRING
81,inserted_at,TIMESTAMP
82,contatos_json,STRING
83,dataModificacao,DATE



🔎 Possíveis colunas de ativo/inativo: ['ativo']

🔎 Possíveis colunas de e-mail FORA do contatos_json: nenhuma coluna óbvia — confira a lista completa acima manualmente

🔎 Possíveis colunas de data de carga/atualização (ETL): nenhuma coluna óbvia — confira a lista completa acima manualmente

--- Distribuição de valores em 'ativo' ---


,valor,qtd
0,True,6686
1,None,8


In [4]:
# ==============================================================================
# DIAGNÓSTICO 2 — QUANTO DE E-MAIL ESTAMOS PERDENDO HOJE?
# ==============================================================================
q_cobertura = f"""
SELECT
    COUNT(*) AS total_empresas,
    COUNTIF(e.contatos_json IS NOT NULL AND e.contatos_json != '') AS com_contatos_json,
    COUNTIF(e.contatos_json IS NULL OR e.contatos_json = '') AS sem_contatos_json__excluidas_hoje,
    COUNTIF(
        e.contatos_json IS NOT NULL AND e.contatos_json != ''
        AND (
            SELECT COUNT(*) FROM UNNEST(JSON_EXTRACT_ARRAY(e.contatos_json)) c
            WHERE JSON_EXTRACT_SCALAR(c, '$.email') IS NOT NULL
              AND JSON_EXTRACT_SCALAR(c, '$.email') != ''
        ) = 0
    ) AS com_contatos_json_mas_sem_email
FROM `{PROJECT_ID}.{DATASET}.{TABELA}` e
"""
print("📊 Cobertura de e-mails na base:")
display(bq_query_to_df(q_cobertura))

print("""
Leitura:
- 'sem_contatos_json__excluidas_hoje' -> empresas 100% fora do relatório atual (gerar_df_completo as descarta).
- 'com_contatos_json_mas_sem_email'   -> aparecem no relatório, mas com a coluna de e-mail vazia.
Se a soma dessas duas colunas for expressiva, "não estar puxando todos os e-mails" é um problema
estrutural do dado de origem (ou de defasagem do 'bronze'), não um bug do SQL em si.
Rode achados_email (diagnóstico 1) para ver se existe e-mail escondido em outra coluna que
o relatório atual ignora por completo.
""")

📊 Cobertura de e-mails na base:


,total_empresas,com_contatos_json,sem_contatos_json__excluidas_hoje,com_contatos_json_mas_sem_email
0,6694,6694,0,2106



Leitura:
- 'sem_contatos_json__excluidas_hoje' -> empresas 100% fora do relatório atual (gerar_df_completo as descarta).
- 'com_contatos_json_mas_sem_email'   -> aparecem no relatório, mas com a coluna de e-mail vazia.
Se a soma dessas duas colunas for expressiva, "não estar puxando todos os e-mails" é um problema
estrutural do dado de origem (ou de defasagem do 'bronze'), não um bug do SQL em si.
Rode achados_email (diagnóstico 1) para ver se existe e-mail escondido em outra coluna que
o relatório atual ignora por completo.



In [ ]:
# ==============================================================================
# CONFIGURAÇÃO DO FILTRO DE ATIVO/INATIVO
# ==============================================================================
# Achado no diagnóstico: coluna 'ativo' é STRING (guarda o texto "True"), não
# BOOLEAN de verdade. SAFE_CAST converte para BOOL sem quebrar em valor inesperado.
# Cliente inativo = NULL (não existe "False" nos dados).
CAMPO_SITUACAO = "ativo"

def _filtro_ativo_sql(apenas_ativos):
    """apenas_ativos: True (só ativos) / False (só inativos, inclui NULL) / None (todos)."""
    if apenas_ativos is None or CAMPO_SITUACAO is None:
        return ""
    if apenas_ativos:
        return f"AND SAFE_CAST(e.{CAMPO_SITUACAO} AS BOOL) IS TRUE"
    else:
        return f"AND SAFE_CAST(e.{CAMPO_SITUACAO} AS BOOL) IS NOT TRUE"

# ==============================================================================
# CONFIGURAÇÃO DO FILTRO DE DIVISÃO
# ==============================================================================
# Achado: 'divisao' é uma STRUCT (não um valor simples), com os campos
# codigoDivisao e descricao. Ex: {'codigoDivisao': '10', 'descricao': 'CLIENTES'}
# Regra de negócio: só considerar clientes das divisões:
#   10 = CLIENTES | 90 = ALESAT | 11 = IPIRANGA | 12 = VIBRA
CAMPO_DIVISAO     = "divisao"
DIVISOES_VALIDAS  = ["10", "90", "11", "12"]

def _filtro_divisao_sql():
    if not DIVISOES_VALIDAS or CAMPO_DIVISAO is None:
        return ""
    valores = ", ".join([f"'{v}'" for v in DIVISOES_VALIDAS])
    return f"AND e.{CAMPO_DIVISAO}.codigoDivisao IN ({valores})"

# ==============================================================================
# EXTRAÇÃO DE E-MAILS — COMPARTILHADA PELA PARTE 1 E PELA PARTE 2
# ==============================================================================
# CONTATO PRINCIPAL: no contatos_json, toda empresa tem exatamente UM contato com
# sequenciaContato = null. É o contato principal do cadastro no CIGAM (os demais
# vêm numerados 1, 2, 3...). Medido em 29/10/2026 nos clientes ativos: 4.680 de
# 4.680 têm um e só um.
#
# O que existia antes e por que não servia:
#   - A coluna 'contato_principal' vinha de e.contato, que é o NOME da pessoa do
#     contato principal (igual ao pessoaContato dele em 100% dos casos), não o
#     e-mail. Está vazia em 2.409 clientes que TÊM e-mail principal, guarda um
#     nome em 335 e, quando guarda e-mail, vem cortada em 40 caracteres.
#   - SELECT DISTINCT dentro de ARRAY() não garante ordem: o principal podia cair
#     em qualquer posição da lista, então não dava para "pegar o primeiro".
#   - Um endereço com ';' no fim ou dois endereços no mesmo campo
#     ("a@x.com;", "a@x.com,b@y.com", "x@y.com>") reprovava na regex e sumia
#     INTEIRO. Eram 16 contatos principais e 66 secundários perdidos assim.
#
# Agora: cada campo de e-mail é quebrado em ; , < > aspas e espaço, cada pedaço
# é validado sozinho, e a lista sai ordenada com o principal primeiro.
# Continua perdido (e não tem como recuperar aqui): endereço truncado na origem,
# tipo "fulano@empresa.com.b" — o CIGAM guardou assim.
#
# PRINCIPAL CONTA SEMPRE COMO ATIVO. O e-mail principal aparecia em
# emails_contatos mas sumia de emails_ativos, porque no CIGAM as flags de
# recebimento do contato principal muitas vezes ficam em N ou em branco
# (nfe=N carta=N bol=vazio). Medido: 72 principais saíam do público ativo
# assim, e em clientes de um contato só isso deixava a empresa sem nenhum
# destinatário. O principal é o contato do cadastro; não é "desligado".
# emails_cobranca continua respeitando a flag de carta de cobrança.
PRINCIPAL_SEMPRE_ATIVO = True

REGEX_EMAIL = r'^[^@\s,;<>"]+@[^@\s,;<>"]+\.[a-z]{2,}$'


def _sql_enderecos(alias: str = "e") -> str:
    """Expressão SQL: ARRAY<STRUCT> com um endereço por item, deduplicado e
    ordenado (principal primeiro, depois na ordem do cadastro).

    Campos: email, ordem, principal, recebe_algo, recebe_carta.
    Um endereço repetido em dois contatos vira um item só; as flags são
    somadas (OR) — se algum dos contatos recebe, o endereço recebe.
    Com PRINCIPAL_SEMPRE_ATIVO, o principal entra em recebe_algo mesmo com
    todas as flags em N/branco.
    """
    principal_ativo = "OR principal" if PRINCIPAL_SEMPRE_ATIVO else ""
    return f"""ARRAY(
      SELECT AS STRUCT
        email,
        MIN(ordem)               AS ordem,
        LOGICAL_OR(principal)    AS principal,
        LOGICAL_OR(recebe_algo {principal_ativo}) AS recebe_algo,
        LOGICAL_OR(recebe_carta) AS recebe_carta
      FROM (
        SELECT
          TRIM(LOWER(pedaco), '.') AS email,
          JSON_EXTRACT_SCALAR(c, '$.sequenciaContato') IS NULL AS principal,
          IF(JSON_EXTRACT_SCALAR(c, '$.sequenciaContato') IS NULL, 0, pos + 1) * 100 + k AS ordem,
          'S' IN (IFNULL(JSON_EXTRACT_SCALAR(c, '$.recebeEmailCartaCobranca'), ''),
                  IFNULL(JSON_EXTRACT_SCALAR(c, '$.recebeEmailCobranca'), ''),
                  IFNULL(JSON_EXTRACT_SCALAR(c, '$.recebeEmailNfe'), ''),
                  IFNULL(JSON_EXTRACT_SCALAR(c, '$.recebeEmailBoleto'), '')) AS recebe_algo,
          IFNULL(JSON_EXTRACT_SCALAR(c, '$.recebeEmailCartaCobranca'), '') = 'S' AS recebe_carta
        FROM UNNEST(JSON_EXTRACT_ARRAY(
               COALESCE(NULLIF({alias}.contatos_json, ''), '[]'))) AS c WITH OFFSET pos,
             UNNEST(SPLIT(REGEXP_REPLACE(
               IFNULL(JSON_EXTRACT_SCALAR(c, '$.email'), ''),
               r'[;,<>"\\s]+', ';'), ';')) AS pedaco WITH OFFSET k
      )
      WHERE REGEXP_CONTAINS(email, r'{REGEX_EMAIL}')
      GROUP BY email
      ORDER BY ordem
    )"""


def _sql_lista(campo_array: str, separador: str, condicao: str = "TRUE") -> str:
    """Expressão SQL: os e-mails de um ARRAY de _sql_enderecos, em ordem, como texto."""
    return f"""ARRAY_TO_STRING(ARRAY(
      SELECT x.email FROM UNNEST({campo_array}) AS x
      WHERE {condicao} ORDER BY x.ordem), '{separador}')"""


In [ ]:
"""
================================================================================
PARTE 1 – CONSULTA PONTUAL DE E-MAILS (v6 – e-mail principal)
Cores: #382fd8 | #1a1a2e | #19e098
Projeto: hip-bonito-453017-m2 | Dataset: bronze | Tabela: cigam__empresas

Instrução: Rode as células de diagnóstico e configuração ANTES desta.
================================================================================
"""

# Não trunca colunas
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

PROJECT_ID = "hip-bonito-453017-m2"
DATASET    = "bronze"
TABELA     = "cigam__empresas"

_client = None

def _get_client():
    global _client
    if _client is None:
        _client = bigquery.Client(project=PROJECT_ID)
    return _client

def _bq_query_to_df(query: str, params=None) -> pd.DataFrame:
    client = _get_client()
    job_config = bigquery.QueryJobConfig(query_parameters=params) if params else None
    rows = client.query(query, job_config=job_config).result()
    return pd.DataFrame([dict(r.items()) for r in rows])


def _colunas_resumo(separador: str) -> str:
    """Colunas de e-mail da busca pontual. contato_principal é o NOME do contato
    principal; o e-mail dele está em email_principal (ver célula de configuração)."""
    enderecos = _sql_enderecos("e")
    return f"""{_sql_lista(enderecos, separador, "x.principal")} AS email_principal,
            {_sql_lista(enderecos, separador)} AS emails_contatos,
            (
                SELECT COUNT(*)
                FROM UNNEST(JSON_EXTRACT_ARRAY(COALESCE(NULLIF(e.contatos_json, ''), '[]'))) AS c
            ) AS total_contatos"""


class ConsultaEmails:
    @staticmethod
    def _por_codigo(codigo: str, separador: str = "; ", apenas_ativos=True) -> pd.DataFrame:
        filtro_ativo = _filtro_ativo_sql(apenas_ativos)
        filtro_divisao = _filtro_divisao_sql()
        query = f"""
        SELECT
            e.codigo AS codigo_cliente,
            e.nomeCompleto AS nome_cliente,
            e.cnpjCpf AS cnpj_cpf,
            e.contato AS contato_principal,
            {_colunas_resumo(separador)}
        FROM `{PROJECT_ID}.{DATASET}.{TABELA}` e
        WHERE e.codigo = @codigo
        {filtro_ativo}
        {filtro_divisao}
        """
        params = [bigquery.ScalarQueryParameter("codigo", "STRING", codigo)]
        return _bq_query_to_df(query, params)

    @staticmethod
    def _por_razao_social(nome: str, separador: str = "; ", apenas_ativos=True) -> pd.DataFrame:
        filtro_ativo = _filtro_ativo_sql(apenas_ativos)
        filtro_divisao = _filtro_divisao_sql()
        query = f"""
        SELECT
            e.codigo AS codigo_cliente,
            e.nomeCompleto AS nome_cliente,
            e.cnpjCpf AS cnpj_cpf,
            e.contato AS contato_principal,
            {_colunas_resumo(separador)}
        FROM `{PROJECT_ID}.{DATASET}.{TABELA}` e
        WHERE (UPPER(e.nomeCompleto) LIKE UPPER(@nome)
           OR UPPER(e.fantasia) LIKE UPPER(@nome))
        {filtro_ativo}
        {filtro_divisao}
        ORDER BY e.nomeCompleto
        LIMIT 50
        """
        params = [bigquery.ScalarQueryParameter("nome", "STRING", f"%{nome}%")]
        return _bq_query_to_df(query, params)

    @staticmethod
    def _por_lista(codigos: list, separador: str = "; ", apenas_ativos=True) -> pd.DataFrame:
        codigos_str = ", ".join([f"'{c}'" for c in codigos])
        filtro_ativo = _filtro_ativo_sql(apenas_ativos)
        filtro_divisao = _filtro_divisao_sql()
        query = f"""
        SELECT
            e.codigo AS codigo_cliente,
            e.nomeCompleto AS nome_cliente,
            e.cnpjCpf AS cnpj_cpf,
            e.contato AS contato_principal,
            {_colunas_resumo(separador)}
        FROM `{PROJECT_ID}.{DATASET}.{TABELA}` e
        WHERE e.codigo IN ({codigos_str})
        {filtro_ativo}
        {filtro_divisao}
        ORDER BY e.codigo
        """
        return _bq_query_to_df(query)

    @staticmethod
    def _detalhe_contatos(codigo: str, apenas_ativos=True) -> pd.DataFrame:
        filtro_ativo = _filtro_ativo_sql(apenas_ativos)
        filtro_divisao = _filtro_divisao_sql()
        query = f"""
        SELECT
            e.codigo AS codigo_cliente,
            e.nomeCompleto AS nome_cliente,
            JSON_EXTRACT_SCALAR(c, '$.sequenciaContato') IS NULL AS principal,
            JSON_EXTRACT_SCALAR(c, '$.pessoaContato') AS pessoa_contato,
            JSON_EXTRACT_SCALAR(c, '$.email') AS email,
            JSON_EXTRACT_SCALAR(c, '$.fone') AS fone,
            JSON_EXTRACT_SCALAR(c, '$.recebeEmailNfe') AS recebe_nfe,
            JSON_EXTRACT_SCALAR(c, '$.recebeEmailCartaCobranca') AS recebe_carta_cobranca,
            JSON_EXTRACT_SCALAR(c, '$.recebeEmailCobranca') AS recebe_cobranca,
            JSON_EXTRACT_SCALAR(c, '$.recebeEmailBoleto') AS recebe_boleto
        FROM `{PROJECT_ID}.{DATASET}.{TABELA}` e,
        UNNEST(JSON_EXTRACT_ARRAY(e.contatos_json)) AS c WITH OFFSET pos
        WHERE e.codigo = @codigo
          AND JSON_EXTRACT_SCALAR(c, '$.email') IS NOT NULL
          AND JSON_EXTRACT_SCALAR(c, '$.email') != ''
        {filtro_ativo}
        {filtro_divisao}
        ORDER BY principal DESC, pos
        """
        params = [bigquery.ScalarQueryParameter("codigo", "STRING", codigo)]
        return _bq_query_to_df(query, params)

    @staticmethod
    def buscar(termo, separador: str = "; ", detalhe: bool = False, apenas_ativos=True) -> pd.DataFrame:
        if isinstance(termo, (list, tuple)):
            return ConsultaEmails._por_lista(list(termo), separador, apenas_ativos=apenas_ativos)
        termo_str = str(termo).strip()
        if detalhe:
            return ConsultaEmails._detalhe_contatos(termo_str, apenas_ativos=apenas_ativos)
        parece_codigo = (
            termo_str.replace(".", "").replace("-", "").replace("/", "").isdigit()
            or (len(termo_str) <= 10 and " " not in termo_str and len(termo_str) >= 3)
        )
        if parece_codigo:
            df = ConsultaEmails._por_codigo(termo_str, separador, apenas_ativos=apenas_ativos)
            if not df.empty:
                return df
            return ConsultaEmails._por_razao_social(termo_str, separador, apenas_ativos=apenas_ativos)
        else:
            return ConsultaEmails._por_razao_social(termo_str, separador, apenas_ativos=apenas_ativos)


# ==============================================================================
# INTERFACE INTERATIVA COM WIDGETS (Paleta da Empresa)
# ==============================================================================

input_termo = widgets.Text(
    value='',
    placeholder='Digite código, nome ou lista separada por vírgula',
    description='Buscar:',
    layout=widgets.Layout(width='500px')
)

check_detalhe = widgets.Checkbox(
    value=False,
    description='Modo detalhe (um contato por linha)',
    indent=False
)

dropdown_situacao = widgets.Dropdown(
    options=[('Somente ativos', True), ('Somente inativos', False), ('Todos', None)],
    value=True,
    description='Situação:',
    layout=widgets.Layout(width='250px')
)

btn_buscar = widgets.Button(
    description='🔍 Buscar',
    button_style='primary',
    layout=widgets.Layout(width='120px')
)

btn_limpar = widgets.Button(
    description='🧹 Limpar',
    button_style='warning',
    layout=widgets.Layout(width='120px')
)

output_area = widgets.Output()
status_label = widgets.HTML(
    value=f'<span style="color:{COR_CINZA}">Pronto para buscar.</span>'
)

def on_buscar_clicked(b):
    with output_area:
        clear_output()
        termo = input_termo.value.strip()
        apenas_ativos = dropdown_situacao.value
        if not termo:
            status_label.value = f'<span style="color:{COR_PRIMARIA}">⚠️ Digite algo para buscar.</span>'
            return
        status_label.value = f'<span style="color:{COR_PRIMARIA}">⏳ Buscando...</span>'
        try:
            if ',' in termo or ';' in termo:
                codigos = [c.strip() for c in termo.replace(';', ',').split(',') if c.strip()]
                df = ConsultaEmails.buscar(codigos, apenas_ativos=apenas_ativos)
                tipo_busca = f"lista com {len(codigos)} códigos"
            else:
                df = ConsultaEmails.buscar(termo, detalhe=check_detalhe.value, apenas_ativos=apenas_ativos)
                tipo_busca = "detalhe" if check_detalhe.value else "automático"

            if df.empty:
                status_label.value = f'<span style="color:{COR_PRIMARIA}">⚠️ Nenhum resultado encontrado para "{termo}".</span>'
                print(f"Nenhum resultado para: {termo}")
            else:
                status_label.value = f'<span style="color:{COR_DESTAQUE}">✅ {len(df)} resultado(s) encontrado(s) ({tipo_busca}).</span>'

                # E-MAILS DESTACADOS — FÁCEIS DE COPIAR
                if not check_detalhe.value and 'emails_contatos' in df.columns:
                    for _, row in df.iterrows():
                        emails = row.get('emails_contatos')
                        principal = row.get('email_principal') or '— sem e-mail no contato principal —'
                        if emails:
                            display(HTML(
                                f"<div style='background:{COR_ESCURA};color:{COR_BRANCO};padding:14px;border-radius:8px;margin:10px 0;border-left:5px solid {COR_DESTAQUE};'>"
                                f"<b style='color:{COR_DESTAQUE};font-size:15px;'>📧 {row.get('nome_cliente', '')}</b> "
                                f"<span style='color:{COR_CINZA};font-size:12px;'>(cód: {row.get('codigo_cliente', '')})</span><br>"
                                f"<span style='color:{COR_CINZA};font-size:12px;'>Principal:</span> "
                                f"<code style='font-size:15px;background:{COR_DESTAQUE};color:{COR_ESCURA};padding:4px 10px;border-radius:6px;display:inline-block;margin-top:8px;'>{principal}</code><br>"
                                f"<span style='color:{COR_CINZA};font-size:12px;'>Todos:</span> "
                                f"<code style='font-size:15px;background:{COR_PRIMARIA};color:{COR_BRANCO};padding:6px 12px;border-radius:6px;display:inline-block;margin-top:8px;'>"
                                f"{emails}"
                                f"</code></div>"
                            ))

                display(HTML(f"<h4 style='color:{COR_PRIMARIA};margin-top:20px;'>📋 Tabela completa ({len(df)} registros)</h4>"))
                display(df)

        except Exception as e:
            status_label.value = f'<span style="color:{COR_PRIMARIA}">❌ Erro: {str(e)}</span>'
            print(f"Erro: {e}")

def on_limpar_clicked(b):
    input_termo.value = ''
    check_detalhe.value = False
    dropdown_situacao.value = True
    with output_area:
        clear_output()
    status_label.value = f'<span style="color:{COR_CINZA}">Pronto para buscar.</span>'

btn_buscar.on_click(on_buscar_clicked)
btn_limpar.on_click(on_limpar_clicked)

box_botoes = widgets.HBox([btn_buscar, btn_limpar])
box_inputs = widgets.VBox([
    widgets.HTML(f"<h3 style='color:{COR_PRIMARIA};margin-bottom:4px;'>🔍 Busca de E-mails de Contatos (CIGAM)</h3>"),
    widgets.HTML(f"<p style='color:{COR_CINZA};margin-top:0;'>Digite um <b style='color:{COR_ESCURA};'>código</b>, <b style='color:{COR_ESCURA};'>razão social</b> ou uma <b style='color:{COR_ESCURA};'>lista</b> (ex: 005875, 000001, 123456)</p>"),
    input_termo,
    widgets.HBox([check_detalhe, dropdown_situacao]),
    box_botoes,
    status_label,
    widgets.HTML("<hr style='border-color:#e5e7eb;margin:12px 0;'>"),
    output_area
])

display(box_inputs)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PARTE 2 – GERAÇÃO EM MASSA + GOOGLE SHEETS (v5)
# ═══════════════════════════════════════════════════════════════════════════════
#
# O QUE MUDOU NA v5 — E-MAIL PRINCIPAL
#
# a. NOVA COLUNA email_principal, preenchida direto do CIGAM: é o e-mail do
#    contato com sequenciaContato nulo (o principal do cadastro). Antes não
#    havia essa coluna; a mais próxima, contato_principal, é o NOME do contato
#    (campo e.contato) e estava vazia em 2.409 clientes que têm e-mail
#    principal — por isso o principal precisava ser mantido à mão e ficava
#    desatualizado. contato_principal continua lá, com o mesmo conteúdo.
#
#    email_principal vai na ÚLTIMA coluna de propósito: a aba é lida por outras
#    rotinas, e inserir no meio deslocaria quem lê por posição. Quem lê pelo
#    cabeçalho (get_all_records, getRange por nome) só precisa trocar para
#    'email_principal'.
#
# b. LISTAS COM O PRINCIPAL PRIMEIRO. emails_contatos / emails_ativos /
#    emails_cobranca saem na ordem do cadastro, principal à frente (quando ele
#    entra naquele público). O DISTINCT antigo embaralhava.
#
# c. ENDEREÇO "SUJO" NÃO SOME MAIS. "a@x.com;", "a@x.com,b@y.com" e
#    "a@x.com>" reprovavam na regex e o contato inteiro sumia (16 principais e
#    66 secundários). Agora cada pedaço é validado sozinho.
#
# d. PRINCIPAL SEMPRE CONTA COMO ATIVO. Ele aparecia em emails_contatos (col. H)
#    mas era descartado de emails_ativos (col. I) quando as flags de
#    recebimento dele no CIGAM estavam em N/branco — 72 casos. Agora entra em
#    emails_ativos e não conta em qtd_desligados. emails_cobranca continua
#    exigindo a flag de carta de cobrança. Ver PRINCIPAL_SEMPRE_ATIVO.
#
# Nada que a v4 trazia deixa de vir: conferido em 29/10/2026 sobre os 4.554
# clientes ativos — zero endereços da v4 ausentes na v5, +40 recuperados.
#
# ───────────────────────────────────────────────────────────────────────────────
# HISTÓRICO v4
#
# O QUE MUDOU EM RELAÇÃO À v3, e por quê:
#
# 1. CNPJ E CÓDIGO VIRAM TEXTO NO SHEETS.
#    set_with_dataframe escreve com USER_ENTERED: o Sheets lê "00242184000104"
#    como número e come os zeros. Na aba atual os comprimentos de cnpj_cpf vão
#    de 1 a 14 dígitos — a ARMAC está gravada como 242184000104. O parâmetro
#    string_escaping resolve, aplicado só às colunas de identificador.
#
# 2. resize=True.
#    worksheet.clear() apaga o conteúdo mas NÃO redimensiona a grade. Se a aba
#    nasceu com 1.000 linhas e o DataFrame tem 4.400, o excedente é descartado
#    sem erro — e, ordenado por nome, some o fim do alfabeto.
#
# 3. E-MAILS DEDUPLICADOS.
#    Sem DISTINCT, o mesmo endereço em dois contatos vira dois envios.
#    Na base: 10.194 contatos resolvem para 6.960 endereços distintos.
#
# 4. TRÊS COLUNAS DE E-MAIL, porque são três públicos diferentes.
#      emails_contatos  todos os endereços válidos
#      emails_ativos    recebe ALGUMA coisa (NFe, boleto, cobrança ou carta)
#      emails_cobranca  só quem recebe carta de cobrança
#
#    Os 416 sem carta de cobrança não são um grupo só. 307 estão desligados de
#    TUDO — sem NFe, sem boleto, sem carta: endereço que a empresa já decidiu
#    não usar, pessoa que saiu, e-mail antigo. Os outros 172 recebem NFe e/ou
#    boleto: gente viva, mantida fora só da conversa de cobrança.
#
#    Para AVISO DE BLOQUEIO o público é emails_ativos, não emails_cobranca:
#    interrupção de serviço é operação, não cobrança, e quem cuida do
#    abastecimento precisa saber que a bomba vai parar. Medido:
#      só carta        9.535 endereços, 88 clientes sem ninguém
#      qualquer flag   9.698 endereços,  3 clientes sem ninguém
#      todos           9.951 endereços,  0 clientes sem ninguém
#    O ganho está quase todo no primeiro passo; o terceiro só ressuscita
#    endereço morto.
#
# 5. FILTRO DE CONTATOS QUE FILTRA DE VERDADE.
#    "contatos_json IS NOT NULL AND != ''" nunca exclui nada: o campo nunca é
#    nulo nem vazio em nenhuma das 4.517 linhas. O filtro real é ter e-mail.
#
# 6. MERGE DE VENDEDOR COM CNPJ NORMALIZADO DOS DOIS LADOS.
#    Antes só o lado do Sheets era limpo com regex; o lado do BigQuery ia como
#    veio. Se o formato divergir, o merge falha em silêncio e TODO mundo sai
#    sem vendedor. Agora ambos passam por dígitos + zfill, e o merge é feito
#    contra CNPJ deduplicado (evita multiplicar linha do cliente).
#
# 7. ERRO DE PERMISSÃO NÃO É MAIS ENGOLIDO.
#    O except Exception devolvia DataFrame vazio e o relatório saía inteiro
#    sem vendedor, parecendo normal.
# ═══════════════════════════════════════════════════════════════════════════════

import re
import time
import gspread
import pandas as pd
from gspread.exceptions import APIError
from gspread_dataframe import set_with_dataframe


def hex_to_rgb_dict(hex_str: str) -> dict:
    """Hexadecimal (#RRGGBB) -> dicionário RGB (0.0 a 1.0) do gspread."""
    hex_str = hex_str.lstrip('#')
    return {'red':   int(hex_str[0:2], 16) / 255.0,
            'green': int(hex_str[2:4], 16) / 255.0,
            'blue':  int(hex_str[4:6], 16) / 255.0}


def _com_retry(func, *args, max_tentativas: int = 5, espera_inicial: float = 2.0, **kwargs):
    """Backoff exponencial para erro transitório do Sheets (429/500/502/503).
    403 e 404 sobem na hora: permissão e planilha errada não melhoram esperando."""
    tentativa = 0
    while True:
        try:
            return func(*args, **kwargs)
        except APIError as e:
            status = getattr(getattr(e, 'response', None), 'status_code', None)
            tentativa += 1
            if status not in (429, 500, 502, 503) or tentativa >= max_tentativas:
                raise
            espera = espera_inicial * (2 ** (tentativa - 1))
            print(f"⚠️ Sheets devolveu {status} ({tentativa}/{max_tentativas}). "
                  f"Aguardando {espera:.0f}s.")
            time.sleep(espera)


# Colunas que são IDENTIFICADOR, não número. Precisam ir como texto ou o
# Sheets come os zeros à esquerda.
COLUNAS_TEXTO = {'codigo_cliente', 'cnpj_cpf', 'cnpj_cpf_limpo', 'fone_principal'}

# REGEX_EMAIL, _sql_enderecos e _sql_lista vêm da célula de configuração.


class GeradorMassaSheets:
    """Gera o relatório completo, cruza com vendedores, formata e envia."""

    @staticmethod
    def gerar_df_completo(separador: str = "; ", apenas_ativos=True,
                          apenas_com_email: bool = True) -> pd.DataFrame:
        """DataFrame das empresas com seus contatos.

        apenas_ativos:    True (só ativos) / False (só inativos) / None (todos)
        apenas_com_email: True exclui quem não tem NENHUM e-mail válido.
                          Antes o filtro olhava contatos_json != '', que nunca
                          exclui nada — o campo nunca vem vazio.
        """
        print("⏳ Consultando o BigQuery...")
        filtro_ativo = _filtro_ativo_sql(apenas_ativos)
        filtro_divisao = _filtro_divisao_sql()

        query = f"""
        WITH base AS (
          SELECT
            e.codigo         AS codigo_cliente,
            e.nomeCompleto   AS nome_cliente,
            e.cnpjCpf        AS cnpj_cpf,
            e.contato        AS contato_principal,
            e.fantasia       AS nome_fantasia,
            e.fone           AS fone_principal,

            -- Um item por endereço, já limpo, deduplicado e com o PRINCIPAL
            -- primeiro (ver _sql_enderecos na célula de configuração)
            {_sql_enderecos("e")} AS enderecos,

            (SELECT COUNT(*)
             FROM UNNEST(JSON_EXTRACT_ARRAY(
                    COALESCE(NULLIF(e.contatos_json, ''), '[]'))) AS c
            ) AS total_contatos
          FROM `{PROJECT_ID}.{DATASET}.{TABELA}` e
          WHERE 1 = 1
          {filtro_ativo}
          {filtro_divisao}
        )
        SELECT
          codigo_cliente, nome_cliente, cnpj_cpf, contato_principal,
          nome_fantasia, fone_principal,
          {_sql_lista("enderecos", separador)}                    AS emails_contatos,
          {_sql_lista("enderecos", separador, "x.recebe_algo")}   AS emails_ativos,
          {_sql_lista("enderecos", separador, "x.recebe_carta")}  AS emails_cobranca,
          ARRAY_LENGTH(enderecos)                                   AS qtd_emails,
          (SELECT COUNT(*) FROM UNNEST(enderecos) x WHERE x.recebe_algo)     AS qtd_ativos,
          (SELECT COUNT(*) FROM UNNEST(enderecos) x WHERE NOT x.recebe_algo) AS qtd_desligados,
          total_contatos,
          {_sql_lista("enderecos", separador, "x.principal")}     AS email_principal
        FROM base
        {"WHERE ARRAY_LENGTH(enderecos) > 0" if apenas_com_email else ""}
        ORDER BY nome_cliente
        """
        df = bq_query_to_df(query)

        # Zeros à esquerda reconstruídos no DataFrame, antes de qualquer escrita.
        if not df.empty:
            df['codigo_cliente'] = (df['codigo_cliente'].astype(str)
                                    .str.replace(r'\D', '', regex=True).str.zfill(6))
            df['cnpj_cpf'] = df['cnpj_cpf'].astype(str).str.replace(r'\D', '', regex=True)
            df['cnpj_cpf'] = df['cnpj_cpf'].apply(
                lambda d: d.zfill(14) if len(d) > 11 else (d.zfill(11) if d else ''))

        if not df.empty:
            print(f"✅ {len(df)} empresa(s) | {int(df['qtd_ativos'].sum())} endereço(s) "
                  f"ativo(s) | {int(df['qtd_desligados'].sum())} desligado(s) de tudo")
            print(f"   sem destinatário ATIVO:   {int((df['emails_ativos'] == '').sum())} "
                  f"empresa(s)  <- é aqui que mora bloqueio sem aviso")
            print(f"   sem destinatário CARTA:   {int((df['emails_cobranca'] == '').sum())} "
                  f"empresa(s)")
            print(f"   sem e-mail PRINCIPAL:     {int((df['email_principal'] == '').sum())} "
                  f"empresa(s)  <- o contato principal no CIGAM está sem e-mail válido")
        return df

    @staticmethod
    def buscar_vendedores(spreadsheet_id_vendedores: str) -> pd.DataFrame:
        """Lê a aba 'Base_Clientes' da planilha de carteira (somente leitura)."""
        print(f"⏳ Lendo vendedores em {spreadsheet_id_vendedores}...")
        spreadsheet = _com_retry(client_gs.open_by_key, spreadsheet_id_vendedores)
        worksheet = _com_retry(spreadsheet.worksheet, "Base_Clientes")
        df_base = pd.DataFrame(_com_retry(worksheet.get_all_records))

        if df_base.empty or 'empresa_cnpj_cpf' not in df_base.columns:
            raise RuntimeError(
                "Base_Clientes vazia ou sem a coluna 'empresa_cnpj_cpf'. "
                "Rode consolidarBaseInteligente() na planilha de carteira. "
                "Sem isso o relatório sairia inteiro sem vendedor, parecendo normal."
            )

        df_base['cnpj_cpf_limpo'] = (df_base['empresa_cnpj_cpf'].astype(str)
                                     .str.replace(r'\D', '', regex=True))
        df_base['cnpj_cpf_limpo'] = df_base['cnpj_cpf_limpo'].apply(
            lambda d: d.zfill(14) if len(d) > 11 else (d.zfill(11) if d else ''))

        # Um CNPJ com dois vendedores multiplicaria a linha do cliente no merge.
        antes = len(df_base)
        df_base = df_base.drop_duplicates(subset=['cnpj_cpf_limpo'], keep='first')
        if len(df_base) < antes:
            print(f"   {antes - len(df_base)} CNPJ(s) duplicado(s) na Base_Clientes; "
                  f"mantida a primeira ocorrência.")

        print(f"✅ {len(df_base)} vendedor(es) por CNPJ.")
        return df_base[['cnpj_cpf_limpo', 'vendedor']]

    @staticmethod
    def enviar_para_sheets(df: pd.DataFrame, spreadsheet_id: str,
                           worksheet_name: str = "Contatos_Emails",
                           limpar_aba: bool = True):
        print(f"⏳ Enviando para {spreadsheet_id} / {worksheet_name}...")
        spreadsheet = _com_retry(client_gs.open_by_key, spreadsheet_id)

        try:
            worksheet = _com_retry(spreadsheet.worksheet, worksheet_name)
            if limpar_aba:
                _com_retry(worksheet.clear)
        except gspread.WorksheetNotFound:
            worksheet = _com_retry(spreadsheet.add_worksheet, title=worksheet_name,
                                   rows=str(max(100, len(df) + 20)),
                                   cols=str(max(26, len(df.columns) + 2)))
            print(f"➕ Aba '{worksheet_name}' criada.")

        df = df.fillna('')
        colunas_texto = [c for c in df.columns if c in COLUNAS_TEXTO]

        def _escapar(valor):
            """Prefixa com ' o que é identificador, para o Sheets guardar como
            TEXTO. Sem isso, '00242184000104' vira o número 242184000104."""
            return bool(re.fullmatch(r'\d+', str(valor)))

        _com_retry(set_with_dataframe, worksheet, df,
                   include_index=False, include_column_header=True,
                   resize=True,                    # <- não remova
                   string_escaping=_escapar)

        _com_retry(worksheet.freeze, rows=1)

        ultima = gspread.utils.rowcol_to_a1(1, len(df.columns))   # funciona além de Z
        _com_retry(worksheet.format, f'A1:{ultima}', {
            'textFormat': {'bold': True,
                           'foregroundColor': hex_to_rgb_dict(COR_DESTAQUE)},
            'backgroundColor': hex_to_rgb_dict(COR_ESCURA),
            'horizontalAlignment': 'CENTER',
        })

        for col in ('emails_contatos', 'emails_ativos', 'emails_cobranca', 'email_principal'):
            if col in df.columns:
                letra = gspread.utils.rowcol_to_a1(1, df.columns.get_loc(col) + 1)[:-1]
                _com_retry(worksheet.format, f'{letra}2:{letra}',
                           {'wrapStrategy': 'WRAP'})

        for col in colunas_texto:
            letra = gspread.utils.rowcol_to_a1(1, df.columns.get_loc(col) + 1)[:-1]
            _com_retry(worksheet.format, f'{letra}2:{letra}',
                       {'numberFormat': {'type': 'TEXT'}})

        print(f"✅ {len(df)} linha(s) enviadas e formatadas.")
        return worksheet

    @classmethod
    def gerar_e_enviar(cls, spreadsheet_id_destino: str,
                       spreadsheet_id_vendedores: str = None,
                       worksheet_name: str = "Contatos_Emails",
                       apenas_ativos=True, apenas_com_email: bool = True):
        spreadsheet_id_vendedores = spreadsheet_id_vendedores or spreadsheet_id_destino

        df_bq = cls.gerar_df_completo(apenas_ativos=apenas_ativos,
                                      apenas_com_email=apenas_com_email)
        df_vend = cls.buscar_vendedores(spreadsheet_id_vendedores)

        # Os dois lados já passaram por dígitos + zfill. Antes só o lado do
        # Sheets era normalizado, e uma divergência de formato zerava o merge
        # inteiro sem levantar erro.
        antes = len(df_bq)
        df = df_bq.merge(df_vend, left_on='cnpj_cpf', right_on='cnpj_cpf_limpo',
                         how='left').drop(columns=['cnpj_cpf_limpo'], errors='ignore')
        assert len(df) == antes, f"merge multiplicou linhas: {antes} -> {len(df)}"

        df['vendedor'] = df['vendedor'].fillna('NÃO MAPEADO')
        casados = int((df['vendedor'] != 'NÃO MAPEADO').sum())
        print(f"✅ Vendedor atribuído a {casados}/{len(df)} "
              f"({100 * casados / max(len(df), 1):.1f}%).")
        if casados == 0:
            raise RuntimeError("NENHUM cliente casou com a base de vendedores. "
                               "Formato de CNPJ divergente — não envie assim.")

        colunas = [c for c in df.columns if c != 'vendedor']
        pos = colunas.index('nome_fantasia') + 1 if 'nome_fantasia' in colunas else len(colunas)
        colunas.insert(pos, 'vendedor')
        df = df[colunas]

        cls.enviar_para_sheets(df, spreadsheet_id_destino, worksheet_name)
        return df


# ═══════════════════════════════════════════════════════════════════════════════
# EXECUÇÃO
# ═══════════════════════════════════════════════════════════════════════════════
ID_DESTINO    = "1KkE6_D_-xug1LSxQYJ9v5vjUtjK72uzawL0pBfC4h3A"
ID_VENDEDORES = "1KSd7cdHUiotmE6WAePq0WnXzhLZz8vggwPkcn1ZGBv4"

df_final = GeradorMassaSheets.gerar_e_enviar(
    spreadsheet_id_destino=ID_DESTINO,
    spreadsheet_id_vendedores=ID_VENDEDORES,
    worksheet_name="Contatos_Emails",
    apenas_ativos=True,
    apenas_com_email=True,
)
df_final.head()